In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
from PIL import Image

from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [3]:

torch.cuda.is_available()

False

In [4]:
file = 'Clothes_Dataset' 

def load_data(file):
    images = []
    labels = []
    
    if not os.path.exists(file):
        print(f"Error: Path {file} not found. Check spelling/case.")
        return None, None, None

    Class = sorted([name for name in os.listdir(file) if os.path.isdir(os.path.join(file, name))])
    label_map = {name: i for i, name in enumerate(Class)}

    for c in Class:
        cls_path = os.path.join(file, c)

        for img_name in os.listdir(cls_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(cls_path, img_name)

                try:
                    img = Image.open(img_path).convert("L")  # grayscale
                    img = img.resize((64, 64))
                    img = np.array(img)

                    images.append(img)
                    labels.append(label_map[c])

                except Exception as e:
                    print(f"Skipping {img_path}: {e}")
                
    return np.array(images), np.array(labels), Class

X, y, classes = load_data(file)

In [28]:
len(classes)

15

In [13]:
mean = X.mean()
std = X.std()
print(f"Mean: {mean:.4f}, Std: {std:.4f}")

Mean: 133.0364, Std: 66.3551


In [14]:
X_norm = (X - mean) / std

In [29]:
class CNN(nn.Module):
    def __init__(self, Layers, Function = nn.ReLU, name = 'CNN-model'):
        super().__init__()
        layers = []

        for item in Layers:
            layers.append(nn.Conv2d(item[0], item[1], kernel_size=3))
            layers.append(Function())

        self.Layers = nn.Sequential(*layers)
        
        self.MaxPool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.25)
        self.Dense = nn.LazyLinear(128)
        self.Output = nn.Linear(128, 15)
        self.Function = Function()
        self.name = name

    def forward(self, x):
        x = self.Layers(x)
        x = self.MaxPool(x)
        x = self.dropout(x)
        x = torch.flatten(x, start_dim=1)
        x = self.Function(self.Dense(x))
        x = self.Output(x)
        return x  

In [31]:
class CNN2(nn.Module):
    def __init__(self, Layers, Function=nn.ReLU, name='CNN-model'):
        super().__init__()
        layers = []

        for item in (Layers):
            layers.append(nn.Conv2d(item[0], item[1], kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(item[1]))
            layers.append(Function())

            if (i + 1) % 2 == 0:
                layers.append(nn.MaxPool2d(2))
                layers.append(nn.Dropout2d(0.25))

        self.Layers = nn.Sequential(*layers)

        self.dropout = nn.Dropout(0.5)
        self.Dense = nn.LazyLinear(128)
        self.Output = nn.Linear(128, 15)
        self.Function = Function()
        self.name = name

    def forward(self, x):
        x = self.Layers(x)
        x = torch.flatten(x, start_dim=1)
        x = self.Function(self.Dense(x))
        x = self.dropout(x)
        x = self.Output(x)
        return x


In [25]:
# train/val split
X_train, X_val, y_train, y_val = train_test_split(X_norm, y, test_size=0.2, stratify=y)

X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train).long()
y_val = torch.tensor(y_val).long()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32, shuffle=False)

In [20]:
model_configs = {
    "Shallow (1 conv)": [(1, 32)],
    "Medium (2 conv)":  [(1, 32), (32, 64)],
    "Deep (3 conv)":    [(1, 32), (32, 64), (64, 128)],
}


In [21]:
def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    epoch_check = {"train_loss": [], "val_f1": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device).view(-1)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        val_f1 = evaluate(model, val_loader)

        epoch_check["train_loss"].append(avg_train_loss)
        epoch_check["val_f1"].append(val_f1)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Macro F1: {val_f1:.4f}")

    return epoch_check


In [30]:
results = {}

for name, layers in model_configs.items():
    print(f"\nTraining {name}")
    model = CNN(layers)
    val_acc = train_model(model, train_loader, val_loader, epochs=10)
    results[name] = val_acc

print("\nFinal Validation Accuracy Comparison:")
for k, v in results.items():
    print(f"{k}: {v:.3f}")


Training Shallow (1 conv)
Epoch 1: Train Acc=0.142, Val Acc=0.203
Epoch 2: Train Acc=0.252, Val Acc=0.302
Epoch 3: Train Acc=0.356, Val Acc=0.359
Epoch 4: Train Acc=0.413, Val Acc=0.381
Epoch 5: Train Acc=0.470, Val Acc=0.390
Epoch 6: Train Acc=0.508, Val Acc=0.402
Epoch 7: Train Acc=0.556, Val Acc=0.431
Epoch 8: Train Acc=0.588, Val Acc=0.435
Epoch 9: Train Acc=0.635, Val Acc=0.441
Epoch 10: Train Acc=0.658, Val Acc=0.434

Training Medium (2 conv)
Epoch 1: Train Acc=0.232, Val Acc=0.387
Epoch 2: Train Acc=0.472, Val Acc=0.445
Epoch 3: Train Acc=0.600, Val Acc=0.459
Epoch 4: Train Acc=0.733, Val Acc=0.465
Epoch 5: Train Acc=0.837, Val Acc=0.445
Epoch 6: Train Acc=0.910, Val Acc=0.481
Epoch 7: Train Acc=0.949, Val Acc=0.491
Epoch 8: Train Acc=0.965, Val Acc=0.492
Epoch 9: Train Acc=0.969, Val Acc=0.488
Epoch 10: Train Acc=0.977, Val Acc=0.491

Training Deep (3 conv)
Epoch 1: Train Acc=0.208, Val Acc=0.373
Epoch 2: Train Acc=0.439, Val Acc=0.459
Epoch 3: Train Acc=0.566, Val Acc=0.467
E